# Wiener Filter:

The Wiener filter is a common algorithm to suppress the noise and a good example for working with the spectrogram.

In [15]:
import numpy as np
from tqdm import tqdm
import sys
sys.path.insert(1, '../Basics')
import WaveFileHandling

x, r = WaveFileHandling.ReadWaveAsNumpyArray('../Audio/P501_D_EN_fm_SWB_48k.wav')
TimeMemoryOfInput = 3

## Implementing STFT and ISTFT
In the following code block, the STFT and its inverse, the ISTFT, are implemented for future usage.

In [16]:
def GetWindowFunction(ws):
    return 0.5 - 0.5 * np.cos(2*np.pi*(np.arange(ws)+0.5)/ws)

class CSpectrogram(object):

    def __init__(self, r):
        self.__r = r

    def GetHopSizeInMilliseconds(self):
        return 10
    
    def GetHopSizeInSamples(self):
        return int(self.__r * self.GetHopSizeInMilliseconds() / 1000)
    
    def GetWindowSizeInSamples(self):
        OverLappingInPercent = 75
        return int(self.GetHopSizeInSamples() / (1 - OverLappingInPercent/100))
    
    def EvalSpectrogram(self, x):    
        hs = self.GetHopSizeInSamples()
        ws = self.GetWindowSizeInSamples()
        w = GetWindowFunction(ws)
        FFTLen = int(2**np.ceil(np.log2(ws)))
        NumberOfBlocks = int((x.shape[0] - ws) / hs) + 1
        X = np.zeros((FFTLen // 2 + 1, NumberOfBlocks), dtype=complex)
        for BlockNumber in range(NumberOfBlocks):
            idx1 = BlockNumber * hs
            idx2 = idx1 + ws
            BlockAnalysis = x[idx1:idx2] * w
            SpectrumAnalysis = np.fft.rfft(BlockAnalysis, n = FFTLen)
            X[:, BlockNumber] = SpectrumAnalysis
        return X

    def EvalTimeDomainSignal(self, X):
        hs = self.GetHopSizeInSamples()
        ws = self.GetWindowSizeInSamples()
        w = GetWindowFunction(ws)
        x = np.zeros(((X.shape[1] - 1) * hs + ws))
        for BlockNumber in range(X.shape[1]):
            idx1 = BlockNumber * hs
            idx2 = idx1 + ws
            BlockSynthesis = np.fft.irfft(X[:, BlockNumber])
            x[idx1:idx2] += BlockSynthesis[:ws] * w * 2 / 3
        return x

ASpectrogram = CSpectrogram(r)
X = ASpectrogram.EvalSpectrogram(x)
y = ASpectrogram.EvalTimeDomainSignal(X)
hs = ASpectrogram.GetHopSizeInSamples()
ws = ASpectrogram.GetWindowSizeInSamples()
x_tmp = x[:y.shape[0]]
x_tmp = x_tmp[ws:-ws]
y_tmp = y[ws:-ws]
SNR = 10 * np.log10(np.sum(x_tmp**2) / np.sum((x_tmp - y_tmp)**2))
assert SNR > 100, f"SNR is too low: {SNR:.2f} dB"

## Noise Model
The noise model is additive white gaussian noise (AWGN):

$y(n)=x(n)+d(n)$

with $x(n)$ being the original (noise free) signal, $d(n)$ being the white gaussian noise and $y(n)$ being the noisy signal.

In [17]:
def ApplyNoise(x):
    return x + np.random.randn(x.shape[0]) * 0.1

## Implementation
The Wiener-Filter in its simplest implementation works in the spectral domain:

$H(f)=\frac{\left|X(f)\right|^2}{\left|X(f)\right|^2+\left|D(f)\right|^2}$

$Z(f) = Y(f)\cdot H(f)$

$\left|X(f)\right|^2$ is the so called power spectral density of $x(n)$.

$\left|D(f)\right|^2$ is the so called power spectral density of the noise signal $d(n)$.

Additionally, it can be assumed, that the signal $x(n)$ and the noise $d(n)$ are uncorrelated. In this case, the following can be shown:

$\left|Y(f)\right|^2 \approx \left|X(f)\right|^2 + \left|D(f)\right|^2$.

This statement is shown, by evaluating the relative error between the true value $\left|Y(f)\right|^2$ and the estimation $\left|X(f)\right|^2 + \left|D(f)\right|^2$.

In [18]:
r = 48000
t = np.arange(r)/r
f = 440
x = np.sin(2*np.pi*f*t)
d = np.random.randn(x.shape[0])*0.1
y = x + d
w = GetWindowFunction(ws)

P_Y = None
idx1 = 0
idx2 = ws
while idx2 < y.shape[0]:
    # evaluate the spectra
    X = np.fft.fft(x[idx1:idx2] * w)
    D = np.fft.fft(d[idx1:idx2] * w)
    Y = np.fft.fft(y[idx1:idx2] * w)

    P_Y_local = (np.abs(Y)**2).reshape((X.shape[0], 1))
    P_Y_hat_local = (np.abs(X)**2 + np.abs(D)**2).reshape((X.shape[0], 1))
    if P_Y is None:
        P_Y = P_Y_local
        P_Y_hat = P_Y_hat_local
    else:
        P_Y = np.concatenate((P_Y, P_Y_local), axis = 1)
        P_Y_hat = np.concatenate((P_Y_hat, P_Y_hat_local), axis = 1)
    
    idx1 += hs
    idx2 += hs

RelativeSquaredError = np.sum((P_Y - P_Y_hat)**2) / np.sum(P_Y**2)
print('relative error = ', RelativeSquaredError)

relative error =  8.51564087560314e-05


From this, the following similarity can be assumed:

$H(f)=\frac{\left|X(f)\right|^2}{\left|X(f)\right|^2+\left|D(f)\right|^2}\approx\frac{\left|X(f)\right|^2}{\left|Y(f)\right|^2}$

In the following, the Wiener Filter is evaluated and applied to spectrograms. The [SNR](../Basics/SignalToNoiseRatio.ipynb) before and after applying the Wiener filter is evaluated.

In [19]:
class CMeanSNREvaluator(object):

    def __init__(self, FilterEstimator):
        self.__MeanSNRWithoutDenoising = 0.0
        self.__MeanSNRwithDenoising = 0.0
        self.__counter = 0
        self.__FilterEstimator = FilterEstimator

    def ProcessSingleFile(self, Filename):
        x, Fs = WaveFileHandling.ReadWaveAsNumpyArray(Filename)
        y = ApplyNoise(x)
        ASpectrogram = CSpectrogram(Fs)
        X = ASpectrogram.EvalSpectrogram(x)
        Y = ASpectrogram.EvalSpectrogram(y)
        X = np.abs(X)
        Y = np.abs(Y)
        DenoiserEstimation = Y * self.__FilterEstimator(X, Y)

        self.__MeanSNRWithoutDenoising += 10*np.log10(np.sum(X**2) / np.sum((X - Y)**2))
        self.__MeanSNRwithDenoising += 10*np.log10(np.sum(X**2) / np.sum((X - DenoiserEstimation)**2))
        self.__counter += 1

    def PrintMeanSNR(self):
        print('Mean SNR without denoising = ', self.__MeanSNRWithoutDenoising / self.__counter)
        print('Mean SNR with denoising = ', self.GetMeanSNRWithDenoising())

    def GetMeanSNRWithDenoising(self):
        return self.__MeanSNRwithDenoising / self.__counter

def FilterEstimator1(X, Y):
    H = (X**2 + 1e-16) / (Y**2 + 1e-16)
    return H

AMeanSNREvaluator = CMeanSNREvaluator(FilterEstimator1)
AMeanSNREvaluator.ProcessSingleFile('../Audio/P501_D_EN_fm_SWB_48k.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/PferdeSchnaubenNichtDieNase.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/SevenEnglishFemale.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/TreeEnglishFemale.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/ZeroEnglishFemale.wav')
AMeanSNREvaluator.PrintMeanSNR()

Mean SNR without denoising =  -2.7432034455813223
Mean SNR with denoising =  9.128788444719508


## Noise Estimation
Unfortunately, in a real measurement environment, the true signal $x$ and the true noise $d(n)$ are unknown. Instead, only the mixture is known:

$y(n)=x(n)+d(n)$

In order to assume the noise level, stationarity of the noise is assumed.
This means, that the power spectra of the noise $\left|D(f)\right|^2$ is nearly constant over time. This assumption is valid, when the noise is not changing a lot over time.
If stationarity of the noise is assumed, the power spectra $\left|D(f)\right|^2$ can be estimated by evaluating the mean value over the columns $t$ of the power spectra $\left|Y(f, t)\right|^2$:

$\left|D(f)\right|^2 \approx \frac{1}{T} \sum_{t=0}^{T-1} \left|Y(f, t)\right|^2$.

From this, the filter can be evaluated by:

$H(f)=\frac{\left|X(f)\right|^2}{\left|X(f)\right|^2 + \left|D(f)\right|^2}=\frac{\left|Y(f)\right|^2 - \left|D(f)\right|^2}{\left|Y(f)\right|^2}=1-\frac{\left|D(f)\right|^2}{\left|Y(f)\right|^2}$

Note: the above given equation only makes sense, if $\left|D(f)\right|^2 < \left|Y(f)\right|^2$ for all $f$.

Therefore, the term $\frac{\left|D(f)\right|}{\left|Y(f)\right|}$ is restricted to the range $0..1$ in the following.

In [20]:
def FilterEstimator2(X, Y):
    D = np.sqrt(np.mean(Y**2, axis = 1))
    D = np.outer(D, np.ones(Y.shape[1]))
    H = 1 - (D**2 + 1e-16) / (Y**2 + 1e-16)
    H = np.maximum(H, 0.0)
    H = np.minimum(H, 1.0)
    return H

AMeanSNREvaluator = CMeanSNREvaluator(FilterEstimator2)
AMeanSNREvaluator.ProcessSingleFile('../Audio/P501_D_EN_fm_SWB_48k.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/PferdeSchnaubenNichtDieNase.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/SevenEnglishFemale.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/TreeEnglishFemale.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/ZeroEnglishFemale.wav')
AMeanSNREvaluator.PrintMeanSNR()

Mean SNR without denoising =  -2.746548066841041
Mean SNR with denoising =  2.5353229670121977


## Fine tuning of Wiener Filter
In the book Digitale Sprachsignalverarbeitung of Prof. Vary et.al., a variation of the Wiener filter is introduced with a tuning parameter $\alpha$:

$H(f)=1-\left(\frac{\left|D(f)\right|^2}{\left|Y(f)\right|^2}\right)^\alpha$

For $\alpha=1$, the Wiener filter is implemented.

For $\alpha=\frac{1}{2}$, the noise spectrum is subtracted.

In the following, a [random walk](../Basics/RandomWalk.ipynb) is implemented, which searches an optimum parameter $\alpha$ resulting in the highest possible SNR for the given trainingsdata.

In [ ]:
def FilterEstimator3(X, Y):
    D = np.sqrt(np.mean(Y**2, axis = 1))
    D = np.outer(D, np.ones(Y.shape[1]))
    H = 1 - ((D**2 + 1e-16) / (Y**2 + 1e-16))**alpha
    H = np.maximum(H, 0.0)
    H = np.minimum(H, 1.0)
    return H

SNR = -np.inf
alpha_opt = 0.464
for n in range(50):
    alpha = alpha_opt
    if n > 0:
        alpha += np.random.randn(1) * 0.1
        alpha = np.abs(alpha[0])
    AMeanSNREvaluator = CMeanSNREvaluator(FilterEstimator3)
    AMeanSNREvaluator.ProcessSingleFile('../Audio/P501_D_EN_fm_SWB_48k.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/PferdeSchnaubenNichtDieNase.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/SevenEnglishFemale.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/TreeEnglishFemale.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/ZeroEnglishFemale.wav')        
    MeanSNRwithDenoising = AMeanSNREvaluator.GetMeanSNRWithDenoising()
    if MeanSNRwithDenoising > SNR:
        SNR = MeanSNRwithDenoising
        alpha_opt = alpha
        print('best SNR = ', SNR)
        print('for alpha = ', alpha_opt)

best SNR =  3.5215531251241456
for alpha =  0.499
best SNR =  3.5284224100420554
for alpha =  0.46432830164433453


## Finetuning of Noise estimation
So far, the noise spectrum $\left|D(f)\right|$ is estimated as the root mean square of the measured noisy spectra:

$\left|D(f)\right| \approx \sqrt{\frac{1}{T} \sum_{t=0}^{T-1} \left|Y(f, t)\right|^2}$

This can be extended to the Hoelder mean of the last spectra:

$\|D(f)\|_p=\left(\frac{1}{T}\sum_{t=0}^{T-1} \left|Y(f, t)\right|^p\right)^{(1/p)}$

The root mean square corresponds to $p=2$.

This extension of the possible parameter space allows a wider search by the [random walk](../Basics/RandomWalk.ipynb) for the best parameters leading to the highest possible SNR regarding the given trainingsdata: 

In [22]:
from scipy.stats import pmean

def FilterEstimator4(X, Y):
    D = pmean(Y, p=p, axis = 1)
    D = np.outer(D, np.ones(Y.shape[1]))
    H = 1 - ((D**2 + 1e-16) / (Y**2 + 1e-16))**alpha
    H = np.maximum(H, 0.0)
    H = np.minimum(H, 1.0)
    return H

SNR = -np.inf
alpha_opt = 0.283
p_opt = -0.359
for n in range(50):
    alpha = alpha_opt
    p = p_opt
    if n > 0:
        alpha += np.random.randn(1)*0.1
        alpha = np.abs(alpha[0])
        p += np.random.randn(1)
        p = p[0]
    AMeanSNREvaluator = CMeanSNREvaluator(FilterEstimator4)
    AMeanSNREvaluator.ProcessSingleFile('../Audio/P501_D_EN_fm_SWB_48k.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/PferdeSchnaubenNichtDieNase.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/SevenEnglishFemale.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/TreeEnglishFemale.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/ZeroEnglishFemale.wav')        
    MeanSNRwithDenoising = AMeanSNREvaluator.GetMeanSNRWithDenoising()
    if MeanSNRwithDenoising > SNR:
        SNR = MeanSNRwithDenoising
        alpha_opt = alpha
        p_opt = p
        print('best SNR = ', SNR)
        print('for alpha = ', alpha_opt)
        print('and p = ', p)

best SNR =  4.797613042265072
for alpha =  0.283
and p =  -0.359


## Programming Exercise:

So far, the Wiener Filter is implemented non-causal: $H(f)$ is evaluated over the whole spectrogram $Y(f,t)$, also for columns $t$ lying in the future.
Implement the causal procedure EvaluateCausalDenoiserEstimation, by analysing only columns of $Y(f,t)$ lying in the past.

In [23]:
def FilterEstimator5(X, Y):    
    H = np.zeros(Y.shape)
    ### solution begins
    for column in range(Y.shape[1]):
        D = np.sqrt(np.mean(Y[:, :column+1]**2, axis = 1))
        H[:, column] = 1 - (D**2+1e-16) / (Y[:, column]**2 + 1e-16)
        H[:, column] = np.maximum(H[:, column], 0.0)
        H[:, column] = np.minimum(H[:, column], 1.0)
    ### solution ends
    return H

X = np.random.randn(500, 100)
Y = np.random.randn(X.shape[0], Y.shape[0])
H1 = FilterEstimator5(X, Y)
RandomColumn = np.random.randint(0, Y.shape[1])
H2 = FilterEstimator2(X, Y[:, :RandomColumn+1])
assert np.allclose(H1[:, RandomColumn], H2[:, RandomColumn]), f"FilterEstimator5 is not correct for column {RandomColumn}"

AMeanSNREvaluator = CMeanSNREvaluator(FilterEstimator5)
AMeanSNREvaluator.ProcessSingleFile('../Audio/P501_D_EN_fm_SWB_48k.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/PferdeSchnaubenNichtDieNase.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/SevenEnglishFemale.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/TreeEnglishFemale.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/ZeroEnglishFemale.wav')
AMeanSNREvaluator.PrintMeanSNR()


Mean SNR without denoising =  -2.7281961059449467
Mean SNR with denoising =  2.3043812616175483


## Exam Preparation
1) Is the proposed Wiener Filter $H(f)=\frac{\left|X(f)\right|^2}{\left|X(f)\right|^2 + \left|D(f)\right|^2}$ complex valued?
2) Evaluate the possible range of values of $H(f)$.
3) Sketch the Wiener Filter $H(f)$ for white noise $\left|D(f)\right|^2=1$, $0<f<10$ and $X(f)=f$.
4) Sketch the SNR depending on $f$ corresponding to the given parameters of task 3).
5) When the Wiener Filter is evaluated, a small positive number $\varepsilon=10^-16$ is added to the denominator and to the numerator. Why?

## Summary
After working with this Jupyter Notebook you should be able to explain the following topics:

- What is the motivation for using the Wiener filter?
- Which noise model / channel model is used for the Wiener filter?
- What is the relationship between the power density spectrum of the noise and the original signal?
- What variants of estimating the unknown noise do you know?